# s03 - Cross-dataset FedAvg: evaluasi R3 (SURVEY)

**Klaim R3** (`paper-survey.md`): FL lintas-dataset sulit/menurun tajam [F11].
Uji dengan **manual feature-name alignment** (bukan SFM). Bandingkan tiga baris:
**centralized** (upper bound), **local-only** (lower bound), **federated** (dari EC2).

**Prasyarat:**
1. `s01` sudah menaruh partisi di S3 `unsw-far/federated_survey/data/{cic,unsw}/`.
2. Run FL (FedAvg, `--mu 0`) sudah SELESAI di EC2 via `aws/fed_node_survey.py`,
   menghasilkan `unsw-far/federated_survey/run_<RUN>/global_final.npz`.

**PROTOKOL EVALUASI (dikoreksi).** Pada EC2, tiap klien melatih dengan **z-score LOKAL**
(statistik dari data train-nya sendiri). Evaluasi harus konsisten dengan itu:
tiap *test set klien dinormalisasi dengan statistik train klien pemiliknya*
(`protocol='local'`), dilaporkan per klien (`cic`, `unsw`) dan gabungan (`union`).
Versi lama mengevaluasi model federated pada test gabungan dengan statistik train
GABUNGAN, yang tidak cocok dengan cara model itu dilatih; hasilnya dipertahankan hanya
sebagai baris diagnostik `pooled(diagnostic)` agar pengaruhnya terlihat.
`centralized` memakai statistik gabungan (`pooled`) karena satu pemilik data.

**Output:** `survey_out/r3_crossdataset.csv` (format panjang: setting, protocol, scope, metrik).
> MLP identik dengan `fed_node_survey.py` (9->128->64->1) agar perbandingan adil.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef, f1_score, recall_score, precision_score
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='unsw-far/federated_survey'
RUN=os.environ.get('RUN','S_R3')   # nama run FedAvg di EC2 (mu=0)
SEED=42
import boto3; s3=boto3.client('s3',region_name=REGION)

## 1. Muat partisi klien dari S3 (hasil s01)

In [ ]:
def load_npz_s3(key):
    try:
        obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e:
        print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
parts={}
for nm in ['cic_train','cic_test','unsw_train','unsw_test']:
    sub='cic' if nm.startswith('cic') else 'unsw'
    d=load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz')
    assert d is not None, f'PRASYARAT GAGAL: {nm}.npz tak ada di S3 (jalankan s01 dulu)'
    parts[nm]=(np.asarray(d['X'],float), np.asarray(d['y'],int))
for k,(X,y) in parts.items():
    print(k, X.shape, 'pos', round(float(y.mean()),3))
# test GLOBAL = gabungan held-out kedua dataset
Xg_te=np.vstack([parts['cic_test'][0], parts['unsw_test'][0]])
yg_te=np.concatenate([parts['cic_test'][1], parts['unsw_test'][1]])
print('GLOBAL test', Xg_te.shape, 'pos', round(float(yg_te.mean()),3))

## 2. MLP numpy IDENTIK `fed_node_survey.py` (training terpusat/lokal untuk baseline)

In [ ]:
LAYERS=[Xg_te.shape[1],128,64,1]
def init_w(seed=42):
    rng=np.random.RandomState(seed); W=[]
    for a,b in zip(LAYERS[:-1],LAYERS[1:]):
        W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(X,a1,z1,a2,z2)
def train(W,X,y,epochs=20,lr=3e-3,batch=256,seed=42):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2)=fwd(W,Xb); g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W
def zfit(X): m=X.mean(0); s=X.std(0); s[s==0]=1; return m,s
def zapp(X,m,s): return (X-m)/s
def metrics(W,m,s,X,y):
    o,_=fwd(W,zapp(X,m,s)); p=(o>=0.5).astype(int)
    return dict(mcc=matthews_corrcoef(y,p), f1=f1_score(y,p,zero_division=0),
                recall=recall_score(y,p,zero_division=0), precision=precision_score(y,p,zero_division=0))

## 3. Baseline: centralized (upper bound) & local-only (lower bound)

- **centralized**: latih di gabungan CIC+UNSW train (hanya pembanding, bukan skenario privasi).
- **local-only**: latih di satu klien saja, uji di GLOBAL test (lower bound tanpa federasi).

In [ ]:
rows = []   # (setting, protocol, scope, metrics)

def mets(y, p):
    return dict(mcc=matthews_corrcoef(y, p), f1=f1_score(y, p, zero_division=0),
                recall=recall_score(y, p, zero_division=0),
                precision=precision_score(y, p, zero_division=0))

def predict_z(W, X, m, s):
    o, _ = fwd(W, zapp(X, m, s))
    return (o >= 0.5).astype(int)

# statistik LOKAL tiap klien = dari TRAIN klien itu sendiri (sama dgn saat training FL di EC2)
LOCAL = {c: zfit(parts[f'{c}_train'][0]) for c in ['cic', 'unsw']}

def eval_protocol(W, norm='local', pooled=None):
    # norm='local'  : tiap test set dinormalisasi dgn statistik klien pemiliknya (konsisten)
    # norm='pooled' : semua test set dinormalisasi dgn satu set statistik gabungan
    ys, ps, out = [], [], {}
    for c in ['cic', 'unsw']:
        Xt, yt = parts[f'{c}_test']
        m, s = LOCAL[c] if norm == 'local' else pooled
        p = predict_z(W, Xt, m, s)
        out[c] = mets(yt, p); ys.append(yt); ps.append(p)
    out['union'] = mets(np.concatenate(ys), np.concatenate(ps))
    return out

def add(setting, protocol, out):
    for scope, m in out.items():
        rows.append((setting, protocol, scope, m))
    print(f'{setting:20s} [{protocol:18s}] ' +
          '  '.join(f"{k}: MCC={v['mcc']:+.3f}" for k, v in out.items()))

# centralized (satu pemilik data -> statistik gabungan adalah normalisasi yang natural)
Xc = np.vstack([parts['cic_train'][0], parts['unsw_train'][0]])
yc = np.concatenate([parts['cic_train'][1], parts['unsw_train'][1]])
mC, sC = zfit(Xc)
WC = train(init_w(SEED), zapp(Xc, mC, sC), yc, epochs=20, seed=SEED)
add('centralized', 'pooled', eval_protocol(WC, 'pooled', (mC, sC)))

# local-only per klien: dilatih dgn statistik lokalnya; tiap test set pakai statistik pemiliknya
for cid in ['cic', 'unsw']:
    Xl, yl = parts[f'{cid}_train']; mL, sL = LOCAL[cid]
    WL = train(init_w(SEED), zapp(Xl, mL, sL), yl, epochs=20, seed=SEED)
    add(f'local-only[{cid}]', 'local', eval_protocol(WL, 'local'))

# KONTROL: centralized dgn normalisasi PER-DOMAIN (tiap sampel di-z-score dgn statistik domain asalnya).
# Memisahkan efek normalisasi per-domain dari efek federasi saat membandingkan dgn federated.
Xcl = np.vstack([zapp(parts['cic_train'][0], *LOCAL['cic']),
                 zapp(parts['unsw_train'][0], *LOCAL['unsw'])])
WCL = train(init_w(SEED), Xcl, yc, epochs=20, seed=SEED)
add('centralized[local-norm]', 'local', eval_protocol(WCL, 'local'))


## 4. Federated: muat `global_final.npz` dari run EC2 (jika ada)
Baris utama = `protocol='local'` (konsisten dengan training EC2 yang memakai z-score
lokal). Baris `pooled(diagnostic)` mereproduksi protokol lama yang TIDAK konsisten dan
hanya untuk memperlihatkan seberapa besar mismatch normalisasi memengaruhi hasil.

In [ ]:
gf = load_npz_s3(f'{BASE}/run_{RUN}/global_final.npz')
if gf is None:
    print('[TBD] global_final.npz belum ada untuk run', RUN, '-> jalankan FedAvg di EC2 dulu.')
else:
    Wf = [gf[f'w{i}'] for i in range(6)]
    add('federated-FedAvg', 'local', eval_protocol(Wf, 'local'))                       # PRIMER
    add('federated-FedAvg', 'pooled(diagnostic)', eval_protocol(Wf, 'pooled', (mC, sC)))  # protokol lama

## 5. Simpan tabel R3 (ANGKA NYATA) -> CSV + unggah S3

In [ ]:
import csv
path = os.path.join(OUTDIR, 'r3_crossdataset.csv')
with open(path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['setting', 'protocol', 'scope', 'mcc', 'f1', 'recall', 'precision'])
    for name, proto, scope, m in rows:
        w.writerow([name, proto, scope, m['mcc'], m['f1'], m['recall'], m['precision']])
print('saved', path)
try:
    s3.upload_file(path, BUCKET, f'{BASE}/results/r3_crossdataset.csv')
    print('upload -> s3://%s/%s/results/r3_crossdataset.csv' % (BUCKET, BASE))
except Exception as e:
    print('(upload dilewati:', e, ')')
print('=== s03 SELESAI. Gunakan baris protocol=local sebagai hasil utama R3. ===')